<a href="https://colab.research.google.com/github/alifihsan7/kelompok9-image-captioning/blob/main/notebooks/03_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/Kelompok9'
import sys
sys.path.append(f'{ROOT}/src')

!nvidia-smi
!ls -R {ROOT} | head -50

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sat Oct 31 09:13:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|          

In [6]:
%%writefile /content/drive/MyDrive/Kelompok9/src/train_utils.py
"""
src/train_utils.py
Modul pelatihan dan evaluasi PyTorch untuk Image Captioning (Kelompok 9).

Sesuai Kontrak Teknis Data V1:
- Input decoder: captions[:, :-1] (panjang T-1 = 29)
- Target label: captions[:, 1:] (panjang T-1 = 29)
- Loss: CrossEntropyLoss(ignore_index=0)
- Perataan dimensi: logits [B * 29, V] dan target [B * 29]
"""

import os
import csv
import time
import random
from pathlib import Path
from typing import Dict, List, Union

import numpy as np
import torch
import torch.nn as nn
from torch.optim import Optimizer
from torch.utils.data import DataLoader


def set_seed(seed: int = 42) -> None:
    """
    Mengunci seluruh generator bilangan acak untuk menjamin reproduksibilitas.

    Args:
        seed (int): Nilai seed acak (default: 42 sesuai kontrak).
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: Optimizer,
    device: torch.device,
    clip: float = 5.0,
) -> float:
    """
    Melatih model selama satu epoch penuh menggunakan skema Teacher Forcing.

    Args:
        model (nn.Module): Model decoder image captioning.
        loader (DataLoader): DataLoader yang menghasilkan batch (features [B, 2048], captions [B, 30]).
        criterion (nn.Module): Fungsi kerugian (nn.CrossEntropyLoss dengan ignore_index=0).
        optimizer (Optimizer): Optimizer pembaruan bobot (Adam).
        device (torch.device): Perangkat komputasi (cuda atau cpu).
        clip (float, optional): Batas ambang maksimum pemotongan gradien (norm clipping). Default: 5.0.

    Returns:
        float: Rata-rata training loss pada epoch tersebut.
    """
    model.train()
    total_loss = 0.0
    num_batches = len(loader)

    for features, captions in loader:
        features = features.to(device, non_blocking=True)
        captions = captions.to(device, non_blocking=True)

        # Penjajaran Teacher Forcing:
        # Input decoder: membuang token terakhir [B, T-1] -> [B, 29]
        # Target label: membuang token awal <start> [B, T-1] -> [B, 29]
        captions_in = captions[:, :-1]
        targets = captions[:, 1:]

        optimizer.zero_grad()

        # Forward pass menghasilkan raw logits [B, 29, V]
        logits = model(features, captions_in)

        # Perataan dimensi sebelum masuk CrossEntropyLoss:
        # logits [B, 29, V] -> [B * 29, V]
        # targets [B, 29] -> [B * 29]
        vocab_size = logits.size(-1)
        loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1))

        # Backward pass & Gradient clipping untuk menjaga kestabilan recurrent gate
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / num_batches if num_batches > 0 else 0.0


def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    """
    Mengevaluasi loss model pada dataset validasi tanpa komputasi gradien.

    Args:
        model (nn.Module): Model decoder image captioning.
        loader (DataLoader): DataLoader validasi (features [B, 2048], captions [B, 30]).
        criterion (nn.Module): Fungsi kerugian (nn.CrossEntropyLoss dengan ignore_index=0).
        device (torch.device): Perangkat komputasi.

    Returns:
        float: Rata-rata validation loss.
    """
    model.eval()
    total_loss = 0.0
    num_batches = len(loader)

    with torch.no_grad():
        for features, captions in loader:
            features = features.to(device, non_blocking=True)
            captions = captions.to(device, non_blocking=True)

            captions_in = captions[:, :-1]
            targets = captions[:, 1:]

            logits = model(features, captions_in)
            vocab_size = logits.size(-1)
            loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1))

            total_loss += loss.item()

    return total_loss / num_batches if num_batches > 0 else 0.0


def fit(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int,
    lr: float,
    device: torch.device,
    exp_name: str,
    root: Union[str, Path] = ".",
) -> Dict[str, List[float]]:
    """
    Loop pelatihan multi-epoch dengan optimizer Adam, penyimpanan checkpoint terbaik,
    serta pencatatan riwayat metrik per epoch ke berkas CSV.

    Args:
        model (nn.Module): Model decoder image captioning.
        train_loader (DataLoader): DataLoader subset training.
        val_loader (DataLoader): DataLoader subset validasi.
        epochs (int): Jumlah iterasi pelatihan.
        lr (float): Learning rate untuk optimizer Adam.
        device (torch.device): Perangkat komputasi (cuda / cpu).
        exp_name (str): Nama eksperimen (misalnya: 'baseline_E0').
        root (Union[str, Path], optional): Jalur folder root proyek Google Drive. Default: '.'.

    Returns:
        Dict[str, List[float]]: Kamus rekapitulasi riwayat metrik selama pelatihan.
    """
    root_path = Path(root)
    checkpoints_dir = root_path / "checkpoints"
    results_dir = root_path / "results"

    checkpoints_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_file = checkpoints_dir / f"best_{exp_name}.pt"
    csv_file = results_dir / f"{exp_name}.csv"

    # Hanya daftarkan parameter yang memerlukan gradien ke Adam
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable_params, lr=lr)

    # Fungsi kerugian standar dengan masking padding token <pad>=0
    criterion = nn.CrossEntropyLoss(ignore_index=0)

    best_val_loss = float("inf")
    history: Dict[str, List[float]] = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "lr": [],
        "seconds": [],
    }

    # Buat berkas CSV baru beserta header kolomnya
    with open(csv_file, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "val_loss", "lr", "seconds"])

    print(f"Mulai eksperimen: {exp_name} | Epochs: {epochs} | LR: {lr} | Target Device: {device}")
    print("-" * 80)

    for epoch in range(1, epochs + 1):
        start_time = time.time()

        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            clip=5.0,
        )

        val_loss = evaluate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
        )

        elapsed_seconds = round(time.time() - start_time, 2)
        current_lr = optimizer.param_groups[0]["lr"]

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["lr"].append(current_lr)
        history["seconds"].append(elapsed_seconds)

        # Tulis entri epoch ke berkas CSV
        with open(csv_file, mode="a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([epoch, f"{train_loss:.4f}", f"{val_loss:.4f}", f"{current_lr:.6f}", elapsed_seconds])

        # Periksa dan simpan model checkpoint terbaik berbasis val_loss terendah
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            checkpoint_payload = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "train_loss": train_loss,
                "lr": current_lr,
                "exp_name": exp_name,
            }
            torch.save(checkpoint_payload, checkpoint_file)
            status_simpan = " [BEST SAVED]"
        else:
            status_simpan = ""

        print(
            f"Epoch [{epoch:02d}/{epochs:02d}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"LR: {current_lr:.6f} | "
            f"Waktu: {elapsed_seconds:>5.2f}s{status_simpan}"
        )

    print("-" * 80)
    print(f"Selesai! Checkpoint terbaik tersimpan di : {checkpoint_file}")
    print(f"Log pelatihan tersimpan di               : {csv_file}")

    return history

Writing /content/drive/MyDrive/Kelompok9/src/train_utils.py


In [10]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path

# Impor modul yang baru saja disimpan
from train_utils import set_seed, train_one_epoch, evaluate, fit

# 1. Kunci seed sesuai kontrak
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Perangkat yang aktif: {device}")

# 2. Definisikan Dummy Model yang menerima (features [B, 2048], captions_in [B, 29]) -> logits [B, 29, V]
class DummyDecoder(nn.Module):
    def __init__(self, vocab_size=3000, feature_dim=2048, hidden_size=512):
        super().__init__()
        self.fc_feat = nn.Linear(feature_dim, hidden_size)
        self.fc_text = nn.Linear(hidden_size, vocab_size)
        self.embedding = nn.Embedding(vocab_size, hidden_size, padding_idx=0)

    def forward(self, features, captions_in):
        # [B, 29] -> [B, 29, 512]
        emb = self.embedding(captions_in)
        # [B, 2048] -> [B, 1, 512]
        feat = self.fc_feat(features).unsqueeze(1)
        # Sinergi dummy fitur visual dan kata
        combined = emb + feat
        logits = self.fc_text(combined)  # [B, 29, vocab_size]
        return logits

# 3. Parameter dummy data
N_train, N_val = 64, 32
B = 16
V = 3000
T = 30  # Panjang sekuens caption asli

# 4. Bangun tensor dummy sesuai kontrak
dummy_train_features = torch.randn(N_train, 2048, dtype=torch.float32)
dummy_train_captions = torch.randint(0, V, (N_train, T), dtype=torch.long)
# Sisipkan token padding (<pad>=0) pada akhir sekuens untuk verifikasi ignore_index
dummy_train_captions[:, 25:] = 0

dummy_val_features = torch.randn(N_val, 2048, dtype=torch.float32)
dummy_val_captions = torch.randint(0, V, (N_val, T), dtype=torch.long)
dummy_val_captions[:, 25:] = 0

train_loader = DataLoader(TensorDataset(dummy_train_features, dummy_train_captions), batch_size=B, shuffle=True)
val_loader = DataLoader(TensorDataset(dummy_val_features, dummy_val_captions), batch_size=B, shuffle=False)

# 5. Uji fungsional loop pelatihan menggunakan fit()
dummy_model = DummyDecoder(vocab_size=V).to(device)
temp_root = Path("./temp_Kelompok9")

history = fit(
    model=dummy_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=2,
    lr=1e-3,
    device=device,
    exp_name="dummy_test",
    root=temp_root
)

# 6. Verifikasi keberadaan output file
checkpoint_path = temp_root / "checkpoints" / "best_dummy_test.pt"
csv_path = temp_root / "results" / "dummy_test.csv"

assert checkpoint_path.is_file(), "File checkpoint gagal dibuat!"
assert csv_path.is_file(), "File CSV riwayat metrik gagal dibuat!"

print("\n✅ Verifikasi modul train_utils.py SUKSES:")
print(f"- Checkpoint tervalidasi di : {checkpoint_path}")
print(f"- CSV tervalidasi di        : {csv_path}")

Perangkat yang aktif: cuda
Mulai eksperimen: dummy_test | Epochs: 2 | LR: 0.001 | Target Device: cuda
--------------------------------------------------------------------------------
Epoch [01/02] | Train Loss: 8.2084 | Val Loss: 8.2336 | LR: 0.001000 | Waktu:  0.64s [BEST SAVED]
Epoch [02/02] | Train Loss: 5.8339 | Val Loss: 8.2320 | LR: 0.001000 | Waktu:  0.02s [BEST SAVED]
--------------------------------------------------------------------------------
Selesai! Checkpoint terbaik tersimpan di : temp_Kelompok9/checkpoints/best_dummy_test.pt
Log pelatihan tersimpan di               : temp_Kelompok9/results/dummy_test.csv

✅ Verifikasi modul train_utils.py SUKSES:
- Checkpoint tervalidasi di : temp_Kelompok9/checkpoints/best_dummy_test.pt
- CSV tervalidasi di        : temp_Kelompok9/results/dummy_test.csv
